# 替え歌動画をColabで生成する

XF MIDI(+編集済みの替え歌JSON)から、NEUTRINOの歌唱合成つき替え歌動画を作ります。

## 事前準備

- ランタイム → ランタイムのタイプを変更 → **GPU(T4)** を選択
- NEUTRINOはセル2が公式配布(Google Drive)から自動ダウンロードします。
  頻繁に使う場合は[公式手順](https://studio-neutrino.com/561/)どおり自分のGoogle Driveの
  `Colab Notebooks/NEUTRINO` に置いておくと2回目以降が速くなります(セル2で切替可)

## 使い方

上から順にセルを実行。途中で `song.mid`(XF形式)をアップロードします。
替え歌の中身を自分で調整したい場合は、[soramimic](https://soramimic.com) の
「MIDIから取り込み → 変換 → 編集ツール → 書き出し」で作ったJSONも一緒にアップロードしてください
(**JSONがあれば変換はブラウザで済んでいるので、このノートは歌わせて動画にするだけ**です)。

何もアップロードしなければ、同梱のサンプル曲(examples/)で一通り試せます。

In [ ]:
# @title 1. セットアップ(2分ほど)
!apt-get -qq update && apt-get -qq install -y ffmpeg fluidsynth fluid-soundfont-gm fonts-noto-cjk > /dev/null
![ -d soramimic-video ] || git clone -q https://github.com/soramimic/soramimic-video.git
!cd soramimic-video && git submodule update --init external/soramimic-wordlists
!cd soramimic-video && pip -q install -e .
print("セットアップ完了")

In [ ]:
# @title 2. NEUTRINOを用意
# 「公式Driveから自動ダウンロード」: 公式配布のNEUTRINO-online-v3.2.2(2.75GB)を
# このVMに直接ダウンロードする(お手軽・毎回ダウンロードが走る)
# 「自分のGoogle Drive」: 事前準備1で置いたNEUTRINOを使う(2回目以降が速い)
import glob
import os

SOURCE = "公式Driveから自動ダウンロード"  # @param ["公式Driveから自動ダウンロード", "自分のGoogle Drive"]
DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/NEUTRINO"  # @param {type:"string"}

if SOURCE == "公式Driveから自動ダウンロード":
    if not glob.glob("/content/npkg/**/bin", recursive=True):
        !pip -q install gdown
        !gdown -q 1oOQOSw9Cha8xVFxsZ-SeXos3e0ZMXe3M -O /content/neutrino.zip
        !unzip -q /content/neutrino.zip -d /content/npkg
    root = glob.glob("/content/npkg/**/bin", recursive=True)[0][: -len("/bin")]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    root = DRIVE_PATH

assert os.path.exists(f"{root}/bin"), f"{root}/bin がありません"
os.environ["NEUTRINO_ROOT"] = root
!chmod -R u+x "$NEUTRINO_ROOT/bin"
print("NEUTRINO_ROOT =", root)
print("モデル一覧:", sorted(os.listdir(f"{root}/model")))

In [ ]:
# @title 3. 入力ファイルをアップロード(キャンセルするとサンプル曲で試せる)
# song.mid(XF形式)、lyrics.txt(元歌詞・任意)、
# soramimic編集ツールで書き出したJSON(任意)をまとめて選択
import os

from google.colab import files

try:
    uploaded = files.upload()
except Exception:
    uploaded = {}
midi_path = next((n for n in uploaded if n.endswith((".mid", ".midi"))), None)
lyrics_path = next((n for n in uploaded if n.endswith(".txt")), None)
editor_json = next((n for n in uploaded if n.endswith(".json")), None)
if midi_path is None:
    midi_path = "soramimic-video/examples/sample_song.mid"
    editor_json = "soramimic-video/examples/sample_editor.json"
    print("アップロードがないので同梱のサンプル曲を使います")
midi_path = os.path.abspath(midi_path)
lyrics_path = os.path.abspath(lyrics_path) if lyrics_path else None
editor_json = os.path.abspath(editor_json) if editor_json else None
print("MIDI:", midi_path, "\n元歌詞:", lyrics_path, "\n編集JSON:", editor_json)

In [ ]:
# @title 4. 変換〜動画生成
import subprocess

WORDLIST = "stations"  # @param ["stations", "baseball", "football", "nations", "pokemon", "physicist", "sekitsui"]
WHERE = "status=current"  # @param {type:"string"}
MODEL = "MERROW"  # @param {type:"string"}

def run(*args):
    print("$", " ".join(args))
    subprocess.run(args, check=True, cwd="soramimic-video")

project = "work/song"
analyze = ["soramimic-video", "analyze", "--midi", midi_path, "--project", project]
if lyrics_path:
    analyze += ["--lyrics", lyrics_path]
run(*analyze)
if editor_json:
    # ブラウザで変換・編集済み: 変換ブリッジ(node+soramimic本体)は不要
    run("soramimic-video", "import-editor", "--project", project, "--file", editor_json)
else:
    # このノート内で自動変換する場合はsoramimic本体のsubmoduleとnodeが必要
    run("git", "submodule", "update", "--init", "external/soramimic")
    subprocess.run("cd soramimic-video/bridge && npm ci --silent", shell=True, check=True)
    run("soramimic-video", "convert", "--project", project,
        "--wordlist", WORDLIST, *(["--where", WHERE] if WHERE else []))
run("soramimic-video", "synthesize", "--project", project, "--model", MODEL)
run("soramimic-video", "mix", "--project", project,
    "--soundfont", "/usr/share/sounds/sf2/FluidR3_GM.sf2")
run("soramimic-video", "video", "--project", project, "--font", "Noto Sans CJK JP")
print("完成: soramimic-video/work/song/video/out.mp4")

In [ ]:
# @title 5. 動画をダウンロード
from google.colab import files

files.download("soramimic-video/work/song/video/out.mp4")
# 画像クレジット(公開時に必要)も一緒にどうぞ
import os

if os.path.exists("soramimic-video/work/song/video/credits.md"):
    files.download("soramimic-video/work/song/video/credits.md")

## トラブルシュート

- **NEUTRINOのコマンドが失敗する**: バージョンによりCLIが異なります。
  `soramimic-video/work/song/neutrino/commands.json` をお使いのNEUTRINOの `Run.sh` に合わせて編集して、セル4を再実行してください
- **日本語字幕が豆腐になる**: セル4の `--font` をインストール済みフォント名に変更
  (`!fc-list :lang=ja` で確認)
- **replace 前の歌詞で歌ってしまう**: 編集JSONの取り込み(セル4のimport-editor)が行われているかログを確認